# rearrange-as-sequential-layer — ex2: Rearrange-based patchify layer inside an nn.Sequential

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rearrange-as-sequential-layer`. Running the final beacon cell reports progress against the `Einops: Rearrange as nn.Sequential layer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange as nn.Sequential layer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rearrange-as-sequential-layer`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rearrange-as-sequential-layer"
DD_SUBTOPIC = "Einops: Rearrange as nn.Sequential layer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Rearrange for patchify — `(b c (h ph) (w pw)) -> b (h w) (ph pw c)`

Ex1 used `Rearrange` as a flatten-to-(B, C·H·W) layer inside `nn.Sequential`. The deepening move is the CANONICAL ViT/transformer patchify rearrange — split an image into non-overlapping patches and flatten each patch.

```python
from einops.layers.torch import Rearrange
patchify = Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)', ph=2, pw=2)
# (B, 3, 8, 8) -> (B, 16, 12)   # 16 patches of (2·2·3) = 12 features
```

**Why the inner ordering `(ph pw c)` matters.** Inside each patch, einops flattens in left-to-right order. `(ph pw c)` walks pixel-by-pixel then channel-by-channel — NHWC ordering. Reordering to `(c ph pw)` gives channel-major, which is what PyTorch tensors use natively but is NOT what a HuggingFace ViT expects. Wrong ordering leads to silent feature-mismatch at the linear projection.

**Why this composes inside `nn.Sequential`.** Same trick as ex1 — `Rearrange` is an `nn.Module`, so it slots between conv stages without a custom forward. The next `Linear(patch_dim, embed_dim)` sees `(B, num_patches, patch_dim)` directly.

### Exercise 2 — Rearrange-based patchify layer inside an nn.Sequential

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the einops patchify pattern `'b c (h ph) (w pw) -> b (h w) (ph pw c)'` as an `einops.layers.torch.Rearrange` layer inside an `nn.Sequential` that maps `(B, 3, H, W)` to `(B, num_patches, embed_dim)`.
> Keywords: rearrange, patchify, vit, sequential
> ```

**KCs targeted:** `patchify-rearrange-pattern`, `rearrange-as-sequential-module`

Implement `ex2_patchify_sequential(in_channels, height, width, patch_size, embed_dim)`. Build a ViT-style patch embedder using ONLY `nn.Sequential` + `einops.layers.torch.Rearrange` + `nn.Linear` — no custom forward.

Inputs (all positional ok):
- `in_channels`: e.g. 3.
- `height, width`: input spatial size; both must be divisible by `patch_size`.
- `patch_size`: side length of a square patch.
- `embed_dim`: output dimensionality per patch.

Pipeline (in order):
1. `Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)', ph=patch_size, pw=patch_size)` — split into non-overlapping patches and flatten EACH patch in NHWC order. Output shape: `(B, num_patches, patch_dim)` where `num_patches = (height // patch_size) * (width // patch_size)` and `patch_dim = patch_size * patch_size * in_channels`.
2. `nn.Linear(patch_dim, embed_dim)` — project each patch to `embed_dim`.

Constraints:
- Return `nn.Sequential`, not a custom Module.
- The model must accept `(B, in_channels, height, width)` and return `(B, num_patches, embed_dim)`.
- DO NOT add extra layers (no LayerNorm, no positional encoding) — this drill is about the Rearrange-as-layer pattern, not the full ViT recipe.

In [ ]:
def ex2_patchify_sequential(in_channels, height, width, patch_size, embed_dim):
    from einops.layers.torch import Rearrange
    patch_dim = patch_size * patch_size * in_channels
    return t.nn.Sequential(
        Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)',
                  ph=patch_size, pw=patch_size),
        t.nn.Linear(patch_dim, embed_dim),
    )


<details><summary>Solution</summary>

```python
def ex2_patchify_sequential(in_channels, height, width, patch_size, embed_dim):
    from einops.layers.torch import Rearrange
    patch_dim = patch_size * patch_size * in_channels
    return t.nn.Sequential(
        Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)',
                  ph=patch_size, pw=patch_size),
        t.nn.Linear(patch_dim, embed_dim),
    )
```

**Why `(ph pw c)` and not `(c ph pw)`.** ViT and most transformer literatures flatten each patch in NHWC order — pixel-by-pixel, channel-last. PyTorch tensors are NCHW natively, so this rearrange does the channel-last conversion as part of the flatten. Reordering to `(c ph pw)` would give channel-major; downstream features would still be patch_dim long but their meaning is permuted.

**The Rearrange-as-Module trick.** `einops.layers.torch.Rearrange` is an `nn.Module`, so it slots into `nn.Sequential` directly. Without it, you'd need a custom Module class just to call `einops.rearrange` inside `forward` — kills the no-boilerplate point.

**`patch_dim = patch_size * patch_size * in_channels`.** Each patch is `ph × pw × c` floats. Always derive it from the inputs; hardcoding `768` or `512` (standard ViT-Base / ViT-Small values) is the trap that breaks the moment you change `in_channels` for grayscale or hyperspectral.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()